# Imports

In [386]:
%matplotlib inline

import os 
import pandas as pd
import numpy as np
import scipy.stats as sst
import matplotlib.pyplot as plt
from collections import defaultdict
import zipfile
import datetime
import re

## Quiz

1. Сколько всего отправленных сообщений? (9831)
2. Дата создания группы? (4 September 2016)
3. Когда было отправлено первое сообщение? (2 October 2016)
4. Расположите участников группы по количеству отправленных сообщений в порядке убывания (от наибольшего значения к наименьшему) (Панас, Тим, Олег Гаврилов, Граф, Токс, Бабокар, ..., Олег Петров, Андрей)
5. Сколько всего названий было  у группы (включая текущее)? (5)
6. Перечислите названия (Я ебу Али Бабу, Я ебу за еду Али Бабу, Я ебу Бабу Егу, Знакомства для секса, Сквиртология {сообщество русских офицеров})
7. Кто добавил описание группы? Назовите описание. (Панас, Два гомосексуалиста и третий их товарищ) 
8. Кто написал больше всего сообщений подряд? (Олег, 34 и Панас, 32 и Васяша, 13)

# Extract

In [218]:
# extract if necessary
with zipfile.ZipFile('WhatsApp Chat - Сквиртология 2.zip', 'r') as f:
    f.extractall()

In [346]:
text_file = open('_chat.txt', 'r', encoding='utf-8')
df = text_file.read()

In [347]:
def whatsapp_extraction(text):
    """ extract information from Сквиртология WhatsApp chat"""
    
    # split by new line
    df = text.split('\n')

    # drop everything other than massages
    df = [line for line in df if (line.startswith('[') or line.startswith('\u200e'))]

    # replace unicode charachters
    for number in range(len(df)):
        df[number] = df[number].replace('\u200e', '')
        df[number] = df[number].replace('\u202a', '')
        df[number] = df[number].replace('\xa0', '-')
        df[number] = df[number].replace('\u202c', '')

    # extract dates
    dates = [x[0] for x in map(lambda x: x.split(', '), df)]
    dates = [x for x in map(lambda x: x.replace('[', ''), dates)]
    no_dates = [x[1] for x in map(lambda x: x.split(', '), df)]

    # extract time
    time = [x[0] for x in map(lambda x: x.split('] '), no_dates)]
    no_time = [x[1] for x in map(lambda x: x.split('] '), no_dates)]

    # extract names
    names = [x[0] for x in map(lambda x: x.split(' ', 1), no_time)]
    names = [x.replace(':', '') for x in names]

    # extract messages
    messages = [x[1] for x in map(lambda x: x.split(' ', 1), no_time)]

    # create df
    df = pd.DataFrame([dates, time, names, messages]).T
    df.columns = ['date', 'time', 'name', 'msg']

    # replace Vasjasha and Oleg
    df.loc[df['msg'].str.contains('Кузнецов:'), 'msg'] = \
    df.loc[df['msg'].str.contains('Кузнецов:'), 'msg'].str.replace('Кузнецов:', '')
    df.loc[df['msg'].str.contains('Петров:'), 'msg'] = \
    df.loc[df['msg'].str.contains('Петров:'), 'msg'].str.replace('Петров:', '')

    
    
    # create timestamp objects
    df.loc[:, 'date'] = pd.to_datetime((df['date'] + ' ' + df['time']))
    df.drop('time', axis=1, inplace=True)
    return df

In [348]:
df = whatsapp_extraction(df)

In [375]:
# list group names
df.loc[df.loc[:, 'msg'].str.contains('изменил'), :]

,date,name,msg
1,2016-04-09 17:49:43,Тим,изменил(-а) тему на “Я ебу Али Бабу”
38,2016-04-10 00:16:06,Тим,изменил(-а) картинку группы
185,2016-09-10 20:46:58,Тим,изменил(-а) тему на “Я ебу за еду Али Бабу”
265,2016-10-17 22:27:44,Тим,изменил(-а) картинку группы
818,2017-04-01 14:15:20,Панас,изменил(-а) тему на “Я ебу Бабу Егу”
2107,2017-10-17 18:49:15,Тим,изменил(-а) тему на “Знакомства для секса”
2618,2017-12-30 22:29:45,Токс,А ты изменился брат
9059,2019-04-10 22:46:26,Панас,изменил(-а) описание группы


In [435]:
name_strike = defaultdict(int)
name_counter = 1
for number in range(len(df['name']) - 1):
    if df.loc[number, 'name'] == df.loc[number + 1, 'name']:
        name_counter += 1
    else:
        if name_strike[df.loc[number, 'name']] < name_counter:
            name_strike[df.loc[number, 'name']] = name_counter
            name_counter = 1
        else:
            name_counter = 1

name_strike

defaultdict(int,
            {'Сквиртология': 1,
             'Тим': 10,
             'Токс': 8,
             'Вы': 1,
             'Олег': 4,
             '+7-915-281‑01‑14': 34,
             'Graf': 6,
             'Панас': 32,
             'Бабокар': 4,
             'Васяша': 13,
             '+7-916-078‑99‑05': 10,
             'Андрей': 2,
             'Саня': 1})